# Feature Engineering — Joint XGBoost + Block Optuna

Uses the **Optuna-tuned XGBoost** setup from `models copy.ipynb`:
- Hyperparameter search space: `build_xgb_classifier()`
- Saved baseline: `optuna_best_extended.json`
- Base feature blocks: `FEATURE_BLOCKS` from models copy

This notebook adds **new feature-engineering blocks** and runs:
1. Fixed-parameter ablation (leave-one-out + add-one-new-block)
2. Joint Optuna: **XGBoost params + all ablation block toggles**

Metric: 5-fold stratified CV log loss (primary), AUC ROC (secondary).

## 1. Imports

Helper modules (same folder as notebook):
- `model_copy_utils.py` — XGB tuning setup from models copy
- `feature_eng_lib.py` — ablation blocks + submission helpers

Run this cell first. Section 8 can run standalone after cells 1, 2, and 4 if `feature_eng_joint_best.json` exists.

In [34]:
import json
import sys
from pathlib import Path

import optuna
import pandas as pd

# Ensure project root is on path when running from notebook
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_copy_utils import (
    CV_FOLDS,
    FEATURE_BLOCKS,
    MODEL_BUILDERS,
    OPTUNA_BEST_PATH,
    OPTUNA_CV_FOLDS,
    OPTUNA_RANDOM_STATE,
    RANDOM_STATE,
    build_tuned_xgb,
    build_xgb_classifier,
    evaluate_xgb_cv,
    get_feature_sets,
    load_data,
    load_optuna_best,
    load_xgb_best,
)
from feature_eng_lib import (
    ALL_ABLATION_BLOCKS,
    BASE_ABLATION_BLOCKS,
    NEW_ABLATION_BLOCKS,
    OPTUNA_N_TRIALS,
    OPTUNA_STORAGE,
    OPTUNA_BEST_PATH as FEATURE_ENG_BEST_PATH,
    all_blocks_active,
    build_feature_cols_from_blocks,
    engineer_all_features,
    evaluate_block_config,
    load_feature_eng_best,
    make_feature_eng_submission,
    make_joint_optuna_objective,
    run_fixed_params_ablation,
    store_feature_eng_best,
    summarize_block_effects,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Load tuned XGBoost from models copy

Reference model: best XGB trial from `models copy.ipynb` (sections 10–12).

In [35]:
xgb_best = load_xgb_best()
XGB_PARAMS = xgb_best["params"]
MODELS_COPY_FEATURE_COLS = xgb_best["feature_cols"]

tuned_xgb = MODEL_BUILDERS["xgboost"](XGB_PARAMS)

train_df = engineer_all_features(load_data("train.csv"))
y = train_df["default"]

drop_cols = ["client_id", "default"]
all_feature_cols = [c for c in train_df.columns if c not in drop_cols]
FEATURE_SETS = get_feature_sets(all_feature_cols)

models_copy_baseline = evaluate_xgb_cv(
    tuned_xgb, train_df[MODELS_COPY_FEATURE_COLS], y
)

print(f"Models copy tuned feature label: {xgb_best['feature_label']}")
print(f"Models copy tuned val log loss: {xgb_best['val_log_loss_mean']:.6f}")
print(f"Models copy feature count: {len(MODELS_COPY_FEATURE_COLS)}")
print(f"XGB params: {json.dumps(XGB_PARAMS, indent=2)}")
print(f"\nRe-eval baseline on current data: {models_copy_baseline['val_log_loss_mean']:.6f}")

Models copy tuned feature label: all_engineered_extended
Models copy tuned val log loss: 0.424062
Models copy feature count: 62
XGB params: {
  "n_estimators": 300,
  "max_depth": 4,
  "learning_rate": 0.021972740562503774,
  "subsample": 0.7758253794788034,
  "colsample_bytree": 0.8970303337810756,
  "reg_lambda": 1.0513964757652297,
  "reg_alpha": 4.319755742829764e-05,
  "min_child_weight": 7
}

Re-eval baseline on current data: 0.424062


## 3. Ablation block definitions

**Base blocks** (from `models copy.ipynb`):
`demographics`, `delay_engineered`, `pay_status`, `pay_amounts`, `bill_amounts`, `credit_util`, `bill_trends`, `models_copy_new_engineered`

**New feature_eng blocks**:
`delay_trends`, `pay_amt_stats`, `payment_change`, `util_stats`, `delay_util_interactions`

In [36]:
print(f"Base ablation blocks ({len(BASE_ABLATION_BLOCKS)}):")
for name, cols in BASE_ABLATION_BLOCKS.items():
    print(f"  {name}: {len(cols)} cols")

print(f"\nNew ablation blocks ({len(NEW_ABLATION_BLOCKS)}):")
for name, cols in NEW_ABLATION_BLOCKS.items():
    print(f"  {name}: {cols}")

full_feature_cols = build_feature_cols_from_blocks(all_blocks_active())
print(f"\nAll blocks ON -> {len(full_feature_cols)} features")

Base ablation blocks (8):
  demographics: 5 cols
  delay_engineered: 5 cols
  pay_status: 6 cols
  pay_amounts: 7 cols
  bill_amounts: 7 cols
  credit_util: 6 cols
  bill_trends: 13 cols
  models_copy_new_engineered: 13 cols

New ablation blocks (5):
  delay_trends: ['recent_delay_mean', 'old_delay_mean', 'delay_deterioration_v2', 'weighted_delay_v2']
  pay_amt_stats: ['mean_pay_amt', 'std_pay_amt', 'max_pay_amt', 'num_zero_payments']
  payment_change: ['recent_pay_mean', 'old_pay_mean', 'payment_change_recent']
  util_stats: ['mean_util', 'max_util', 'std_util', 'months_high_util', 'months_over_limit', 'recent_util_vs_avg']
  delay_util_interactions: ['recent_delay_x_util', 'delay_count_x_util', 'severe_delay_x_util']

All blocks ON -> 82 features


## 4. Fixed-parameter ablation (models copy XGB params)

Uses tuned hyperparameters from section 2; only feature blocks vary.

- **Leave-one-out:** all blocks ON, remove one block at a time
- **Add-one-new:** base blocks only, add one new block at a time

Negative `delta_val_log_loss` = improvement.

In [37]:
leave_one_out_report, add_one_new_report = run_fixed_params_ablation(
    train_df, y, XGB_PARAMS
)

print("Leave-one-out ablation (reference: all blocks ON):")
print(leave_one_out_report.to_string(index=False))

print("\nAdd-one-new-block ablation (reference: base blocks only):")
print(add_one_new_report.to_string(index=False))

Leave-one-out ablation (reference: all blocks ON):
                             scenario              block_changed  val_log_loss_mean  val_roc_auc_mean  delta_val_log_loss  n_features
              leave_out: delay_trends               delay_trends           0.423923          0.788503           -0.000248          78
          leave_out: delay_engineered           delay_engineered           0.423942          0.788593           -0.000228          77
               leave_out: bill_trends                bill_trends           0.423990          0.787774           -0.000180          69
   leave_out: delay_util_interactions    delay_util_interactions           0.423994          0.788226           -0.000177          79
            leave_out: payment_change             payment_change           0.424094          0.788518           -0.000076          79
                        all_blocks_on                                      0.424171          0.788309            0.000000          82
           

## 5. Optuna setup — joint params + blocks

Search space per trial:
- **Hyperparameters:** same ranges as `build_xgb_classifier()` in models copy
- **Feature blocks:** on/off toggle for each block in `ALL_ABLATION_BLOCKS`

Uses a separate SQLite DB so it does not conflict with `models copy` studies.

In [38]:
feature_eng_study = optuna.create_study(
    study_name="xgb_joint_params_and_blocks",
    storage=OPTUNA_STORAGE,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_RANDOM_STATE),
)

joint_objective = make_joint_optuna_objective(train_df, y)

print(f"Optuna storage: {OPTUNA_STORAGE}")
print(f"Study: {feature_eng_study.study_name}")
print(f"Planned trials: {OPTUNA_N_TRIALS}")
print(f"Block toggles: {len(ALL_ABLATION_BLOCKS)}")
print(f"XGB params tuned: n_estimators, max_depth, learning_rate, subsample,")
print(f"                  colsample_bytree, reg_lambda, reg_alpha, min_child_weight")

Optuna storage: sqlite:///optuna_feature_eng_joint.db
Study: xgb_joint_params_and_blocks
Planned trials: 50
Block toggles: 13
XGB params tuned: n_estimators, max_depth, learning_rate, subsample,
                  colsample_bytree, reg_lambda, reg_alpha, min_child_weight


## 6. Run joint Optuna search (manual)

Run when ready — can take several minutes.

In [39]:
feature_eng_study.optimize(
    joint_objective,
    n_trials=200,
    show_progress_bar=True,
)

best = feature_eng_study.best_trial
print(f"Best val log loss: {best.value:.6f}")
print(f"Best enabled blocks: {best.user_attrs['enabled_blocks']}")
print(f"Best XGB params: {best.user_attrs['xgb_params']}")

Best trial: 449. Best value: 0.422981: 100%|██████████| 200/200 [16:24<00:00,  4.92s/it]

Best val log loss: 0.422981
Best enabled blocks: ['demographics', 'delay_engineered', 'pay_status', 'bill_amounts', 'models_copy_new_engineered', 'delay_trends', 'payment_change', 'util_stats']
Best XGB params: {'n_estimators': 350, 'max_depth': 5, 'learning_rate': 0.01898661236268957, 'subsample': 0.7868082902864677, 'colsample_bytree': 0.8210052558875973, 'reg_lambda': 1.9483028639638904, 'reg_alpha': 1.0030905328706748e-06, 'min_child_weight': 7}


## 7. Block impact report (from joint Optuna trials)

For each block: mean val log loss when block is **ON** vs **OFF** across all completed trials.

- Negative `delta_on_minus_off` → block helps when enabled
- Compare `delta_vs_reference_when_on` to models copy baseline from section 2

In [40]:
reference_val = models_copy_baseline["val_log_loss_mean"]
block_effects = summarize_block_effects(feature_eng_study, reference_val)

FEATURE_ENG_BEST = store_feature_eng_best(feature_eng_study, reference_val)

comparison = pd.DataFrame(
    [
        {
            "source": "models_copy_tuned_baseline",
            "val_log_loss_mean": reference_val,
            "val_roc_auc_mean": models_copy_baseline["val_roc_auc_mean"],
            "n_features": len(MODELS_COPY_FEATURE_COLS),
        },
        {
            "source": "feature_eng_joint_best",
            "val_log_loss_mean": FEATURE_ENG_BEST["best_val_log_loss_mean"],
            "val_roc_auc_mean": FEATURE_ENG_BEST["best_val_roc_auc_mean"],
            "n_features": FEATURE_ENG_BEST["n_features"],
        },
    ]
)

print(f"Saved {FEATURE_ENG_BEST_PATH}")
print("\nModels copy baseline vs feature_eng joint best:")
print(comparison.to_string(index=False))

print("\nPer-block effects (negative delta_on_minus_off => block helps):")
print(block_effects.to_string(index=False))

block_effects

Saved feature_eng_joint_best.json

Models copy baseline vs feature_eng joint best:
                    source  val_log_loss_mean  val_roc_auc_mean  n_features
models_copy_tuned_baseline           0.424062          0.788286          62
    feature_eng_joint_best           0.422981          0.789046          49

Per-block effects (negative delta_on_minus_off => block helps):
                     block block_group  trials_with_block_on  trials_with_block_off  mean_val_log_loss_block_on  mean_val_log_loss_block_off  delta_on_minus_off  improves_when_on  delta_vs_reference_when_on
models_copy_new_engineered        base                   486                     14                    0.424491                     0.434628           -0.010137              True                    0.000429
                util_stats feature_eng                   485                     15                    0.424497                     0.433758           -0.009260              True                    0.000435
   

,block,block_group,trials_with_block_on,trials_with_block_off,mean_val_log_loss_block_on,mean_val_log_loss_block_off,delta_on_minus_off,improves_when_on,delta_vs_reference_when_on
7,models_copy_new_engineered,base,486,14,0.424491,0.434628,-0.010137,True,0.000429
11,util_stats,feature_eng,485,15,0.424497,0.433758,-0.009260,True,0.000435
2,pay_status,base,485,15,0.424542,0.432317,-0.007775,True,0.000480
0,demographics,base,481,19,0.424510,0.431496,-0.006986,True,0.000447
4,bill_amounts,base,484,16,0.424572,0.430937,-0.006365,True,0.000509
8,delay_trends,feature_eng,354,146,0.424358,0.425786,-0.001427,True,0.000296
1,delay_engineered,base,224,276,0.424473,0.425021,-0.000547,True,0.000411
12,delay_util_interactions,feature_eng,218,282,0.424819,0.424741,0.000078,False,0.000757
6,bill_trends,base,230,270,0.425082,0.424514,0.000568,False,0.001020
10,payment_change,feature_eng,79,421,0.425280,0.424681,0.000599,False,0.001217


## 8. Test submission (best joint combination)

Train tuned XGB on full train data with best feature cols + params; write `submission_feature_eng.csv`.

**Minimal run:** sections 1 → 2 → 4 → 8 (requires `feature_eng_joint_best.json` from section 7, or from a prior run).

In [41]:
# Uses FEATURE_ENG_BEST from section 7 if available, else loads feature_eng_joint_best.json
if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

test_df = engineer_all_features(load_data("test.csv"))

submission_feature_eng = make_feature_eng_submission(
    train_df,
    test_df,
    y,
    best_config=FEATURE_ENG_BEST,
    output_path="submission_feature_eng.csv",
)

submission_feature_eng.head()

Saved submission_feature_eng.csv (6000 rows, mean prob=0.219231, 49 features)


,client_id,default_probability
0,CC_0012A082B7B7,0.134313
1,CC_0012BC27DFB6,0.150634
2,CC_001563B2143D,0.100019
3,CC_001B46930B6F,0.083934
4,CC_001D771E1C9F,0.037716


## 9. Hard-error analysis (tuned XGB baseline)

Uses **feature_eng joint best** features + params (`feature_eng_joint_best.json`). Does not change sections 1–8 results.

- Fixed 5-fold `StratifiedKFold` splits (`random_state=42`)
- Confidently wrong: misclassified with confidence > 0.9
- Compare **confident false negatives** vs **correctly predicted defaults**
- Investigate unstable `bill_pct_change` features

In [42]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import feature_eng_lib
importlib.reload(feature_eng_lib)

from feature_eng_lib import (
    TARGETED_BLOCKS,
    assign_error_groups,
    compute_oof_with_folds,
    engineer_targeted_features,
    load_feature_eng_best,
    prediction_confidence,
    standardized_mean_differences,
    summarize_oof_metrics,
)

sns.set_theme(style="whitegrid")
CONFIDENCE_THRESHOLD = 0.9

if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

BASELINE_FEATURE_COLS = FEATURE_ENG_BEST["best_feature_cols"]
JOINT_XGB_PARAMS = FEATURE_ENG_BEST["best_xgb_params"]

train_targeted_df = engineer_targeted_features(load_data("train.csv"))
y = train_targeted_df["default"].astype(int)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(cv.split(train_targeted_df[BASELINE_FEATURE_COLS], y))

baseline_oof, baseline_fold_rows = compute_oof_with_folds(
    train_targeted_df[BASELINE_FEATURE_COLS],
    y,
    FOLDS,
    JOINT_XGB_PARAMS,
)
baseline_hard_error_metrics = summarize_oof_metrics(
    y,
    baseline_oof,
    baseline_fold_rows,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

print(f"Baseline features: {len(BASELINE_FEATURE_COLS)}")
print(f"Saved joint-best val log loss (section 7): {FEATURE_ENG_BEST['best_val_log_loss_mean']:.6f}")
print(f"Section 9 fold-OOF val log loss: {baseline_hard_error_metrics['val_log_loss_mean']:.6f}")
print(f"Confident FN count (conf > {CONFIDENCE_THRESHOLD}): {baseline_hard_error_metrics['confident_fn_count']}")

ImportError: cannot import name 'assign_error_groups' from 'feature_eng_lib' (c:\Users\Nick\PycharmProjects\interuni_datathon_2026\feature_eng_lib.py)

In [ ]:
analysis_df = train_targeted_df[["client_id", "default"] + BASELINE_FEATURE_COLS].copy()
analysis_df["oof_prob"] = baseline_oof
analysis_df["confidence"] = prediction_confidence(baseline_oof)
analysis_df["pred_class"] = (baseline_oof >= 0.5).astype(int)
analysis_df["error_group"] = assign_error_groups(
    y, baseline_oof, confidence_threshold=CONFIDENCE_THRESHOLD
)

group_counts = analysis_df["error_group"].value_counts()
print("Prediction groups (confidence > 0.9 for confident errors):")
print(group_counts.to_string())

error_summary = (
    analysis_df.groupby("error_group")
    .agg(
        count=("client_id", "count"),
        mean_confidence=("confidence", "mean"),
        default_rate=("default", "mean"),
    )
    .sort_values("count", ascending=False)
)
print("\nGroup summary:")
print(error_summary.round(4).to_string())

rank_cols = [
    "client_id", "default", "oof_prob", "pred_class", "confidence",
    "error_group", "PAY_0", "max_delay", "mean_util", "bill_pct_change_1_2",
]
rank_cols = [c for c in rank_cols if c in analysis_df.columns]

for group_name in ["confident_fn", "confident_fp"]:
    subset = analysis_df[analysis_df["error_group"] == group_name].nlargest(10, "confidence")
    print(f"\nTop 10 {group_name}:")
    print(subset[rank_cols].to_string(index=False))

NameError: name 'train_targeted_df' is not defined

In [ ]:
confident_fn_df = analysis_df[analysis_df["error_group"] == "confident_fn"]
correct_default_df = analysis_df[analysis_df["error_group"] == "correct_default"]

fn_vs_correct_shift = standardized_mean_differences(
    confident_fn_df,
    correct_default_df,
    BASELINE_FEATURE_COLS,
)
fn_shift_table = (
    pd.DataFrame(
        {
            "feature": fn_vs_correct_shift.index,
            "std_shift_vs_correct_default": fn_vs_correct_shift.values,
        }
    )
    .assign(abs_shift=lambda d: d["std_shift_vs_correct_default"].abs())
    .sort_values("abs_shift", ascending=False)
)

print(
    f"Confident FN ({len(confident_fn_df):,}) vs correctly predicted defaults "
    f"({len(correct_default_df):,})"
)
print("\nTop 20 standardized mean shifts:")
print(fn_shift_table.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
plot_data = fn_shift_table.head(15).sort_values("std_shift_vs_correct_default")
colors = [
    "#d62728" if x > 0 else "#1f77b4"
    for x in plot_data["std_shift_vs_correct_default"]
]
ax.barh(plot_data["feature"], plot_data["std_shift_vs_correct_default"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Std shift (confident FN − correct default)")
ax.set_title("Top 15 feature shifts: confident FN vs correct defaults")
plt.tight_layout()
plt.show()

In [ ]:
raw_pct_cols = [c for c in BASELINE_FEATURE_COLS if c.startswith("bill_pct_change_")]
robust_pct_cols = TARGETED_BLOCKS["robust_bill_pct"]

instability_rows = []
for raw_col in raw_pct_cols:
    suffix = raw_col.replace("bill_pct_change_", "")
    robust_col = f"bill_pct_change_{suffix}_robust"
    clip_col = f"bill_pct_change_{suffix}_clip"
    raw_vals = train_targeted_df[raw_col]
    instability_rows.append(
        {
            "feature": raw_col,
            "max_abs_raw": float(raw_vals.abs().max()),
            "p99_abs_raw": float(raw_vals.abs().quantile(0.99)),
            "max_abs_robust": float(train_targeted_df[robust_col].abs().max()),
            "max_abs_clip": float(train_targeted_df[clip_col].abs().max()),
        }
    )

instability_table = pd.DataFrame(instability_rows).sort_values("max_abs_raw", ascending=False)
print("Bill pct-change instability (raw vs robust vs clipped):")
print(instability_table.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
example_col = "bill_pct_change_1_2"
example_robust = "bill_pct_change_1_2_robust"
example_clip = "bill_pct_change_1_2_clip"

sns.histplot(train_targeted_df[example_col].clip(-20, 20), bins=40, ax=axes[0], color="#d62728")
axes[0].set_title(f"{example_col} (clipped display ±20)")
axes[0].set_xlabel("raw pct change")

compare_df = pd.DataFrame(
    {
        "robust": train_targeted_df[example_robust],
        "clipped": train_targeted_df[example_clip],
    }
).melt(var_name="variant", value_name="value")
sns.kdeplot(data=compare_df, x="value", hue="variant", ax=axes[1])
axes[1].set_title(f"Robust vs clipped alternatives ({example_col})")
plt.tight_layout()
plt.show()

## 10. Targeted feature engineering + sequential block test

Adds features aimed at **defaults that look superficially low-risk**, then tests blocks **sequentially** against the section 9 baseline using the same `FOLDS`.

**Blocks (in order):**
1. `robust_bill_pct` — denominators `abs(BILL_AMT)+1000`, clipped pct changes
2. `pay_amt_stats` — zero-payment count, pay mean/std/max
3. `payment_change` — recent vs old pay means, payment deterioration
4. `stabilized_pay_bill` — pay/bill ratios with stabilized denominators
5. `low_risk_interactions` — low-util × high-limit, limit/util, pay stress interactions

Keep a block only if OOF log loss improves. No threshold or class-weight tuning.

In [ ]:
importlib.reload(feature_eng_lib)
from feature_eng_lib import run_sequential_targeted_block_search, TARGETED_BLOCK_ORDER

print("Targeted blocks to test:")
for block_name in TARGETED_BLOCK_ORDER:
    print(f"  {block_name}: {TARGETED_BLOCKS[block_name]}")

comparison_df, best_feature_cols, enabled_blocks = run_sequential_targeted_block_search(
    train_targeted_df,
    y,
    BASELINE_FEATURE_COLS,
    JOINT_XGB_PARAMS,
    FOLDS,
    block_order=TARGETED_BLOCK_ORDER,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

display_cols = [
    "rank",
    "scenario",
    "block_added",
    "kept",
    "n_features",
    "val_log_loss_mean",
    "val_log_loss_std",
    "val_roc_auc_mean",
    "train_val_log_loss_gap",
    "confident_fn_count",
    "delta_confident_fn",
]
print("\nRanked comparison (baseline + each block attempt):")
print(comparison_df[display_cols].round(6).to_string(index=False))

print(f"\nBlocks kept: {enabled_blocks if enabled_blocks else '(none)'}")
print(f"Best feature count: {len(best_feature_cols)}")
print(f"Best val log loss: {comparison_df['val_log_loss_mean'].min():.6f}")

In [ ]:
added_cols = [c for c in best_feature_cols if c not in BASELINE_FEATURE_COLS]

best_feature_set = {
    "baseline_feature_cols": BASELINE_FEATURE_COLS,
    "enabled_targeted_blocks": enabled_blocks,
    "added_feature_cols": added_cols,
    "best_feature_cols": best_feature_cols,
    "xgb_params": JOINT_XGB_PARAMS,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "baseline_confident_fn_count": int(baseline_hard_error_metrics["confident_fn_count"]),
}

print("Best feature set after sequential targeted block search:")
print(f"  Baseline features: {len(BASELINE_FEATURE_COLS)}")
print(f"  Added features: {len(added_cols)}")
print(f"  Total best features: {len(best_feature_cols)}")
print(f"  Enabled blocks: {enabled_blocks}")

if added_cols:
    print("\nNew features kept:")
    for col in added_cols:
        print(f"  - {col}")
else:
    print("\nNo targeted block improved OOF log loss; baseline feature set remains best.")

best_feature_set